In [3]:
%run init_env.py

Added to Python path: C:\git\cs5305\genai-langchain
Environment initialization completed successfully.


## Summarize messages

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

llm_std = create_azure_llm("chat")
llm_nano = create_azure_llm("nano")

agent = create_agent(
    model=llm_std,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm_nano,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

Creating Azure OpenAI LLM
  Deployment: lums-gpt-4.1-mini
  Endpoint: https://aoai-foundry-swc.openai.azure.com/
  API Version: 2025-01-01-preview
  Temperature: 0.7
Creating Azure OpenAI LLM
  Deployment: gpt-5.4-nano
  Endpoint: https://aoai-foundry-swc.openai.azure.com/
  API Version: 2025-01-01-preview
  Temperature: 0.7


In [ ]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nDetermine and answer a sequence of questions about a fictional place called “Lunapolis” (capital of the moon, weather, population of cheese miners, and whether the cheese miners’ union will strike).\n\n## SUMMARY\n- Q: “What is the capital of the moon?”  \n  - A: “Lunapolis.”\n- Q: “What is the weather in Lunapolis?”  \n  - A: “Skies are clear, with a high of 120C and a low of -100C.”\n- Q: “How many cheese miners live in Lunapolis?”  \n  - A: “100,000 cheese miners.”\n- Q: “Do you think the cheese miners' union will strike?”  \n  - A: “Yes, because they are unhappy with the new president.”\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='e38f586b-213e-47a2-93ce-86bd0754c0c1'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", addition

Failed to get info from https://eu.api.smith.langchain.com: LangSmithConnectionError('Connection error caused failure to GET /info in LangSmith API. Please confirm your LANGCHAIN_ENDPOINT. ConnectTimeout(MaxRetryError("HTTPSConnectionPool(host=\'eu.api.smith.langchain.com\', port=443): Max retries exceeded with url: /info (Caused by ConnectTimeoutError(<HTTPSConnection(host=\'eu.api.smith.langchain.com\', port=443) at 0x1f9539ed970>, \'Connection to eu.api.smith.langchain.com timed out. (connect timeout=10.0)\'))"))\nContent-Length: None\nAPI Key: lsv2_********************************************39')
Run compression is not enabled. Please update to the latest version of LangSmith. Falling back to regular multipart ingestion.


In [6]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
Determine and answer a sequence of questions about a fictional place called “Lunapolis” (capital of the moon, weather, population of cheese miners, and whether the cheese miners’ union will strike).

## SUMMARY
- Q: “What is the capital of the moon?”  
  - A: “Lunapolis.”
- Q: “What is the weather in Lunapolis?”  
  - A: “Skies are clear, with a high of 120C and a low of -100C.”
- Q: “How many cheese miners live in Lunapolis?”  
  - A: “100,000 cheese miners.”
- Q: “Do you think the cheese miners' union will strike?”  
  - A: “Yes, because they are unhappy with the new president.”

## ARTIFACTS
None

## NEXT STEPS
None


Failed to multipart ingest runs: Connection error caused failure to POST https://eu.api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your LANGCHAIN_ENDPOINT. ConnectTimeout(MaxRetryError("HTTPSConnectionPool(host='eu.api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by ConnectTimeoutError(<HTTPSConnection(host='eu.api.smith.langchain.com', port=443) at 0x1f97f2a3980>, 'Connection to eu.api.smith.langchain.com timed out. (connect timeout=3)'))"))
Content-Length: 38799
API Key: lsv2_********************************************39trace=019d7b26-2e6c-7e50-bf4a-885216732e03,id=019d7b26-2e6c-7e50-bf4a-885216732e03; trace=019d7b26-2e6c-7e50-bf4a-885216732e03,id=019d7b26-2f6d-7c50-b3c3-2b3cb4727102; trace=019d7b26-2e6c-7e50-bf4a-885216732e03,id=019d7b26-2f6e-7fc0-b99c-ad39a4c33264; trace=019d7b26-2e6c-7e50-bf4a-885216732e03,id=019d7b26-43f5-7ca0-8d69-042b0882fe32; trace=019d7b26-2e6c-7e50-bf4a-885216732e03,id=019d7b26-43f6-7de2-8b

## Trim/delete messages

In [7]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [8]:
agent = create_agent(
    model=llm_std,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [9]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='0a468eca-c95b-46d6-bcc3-e6a41c8fc1b0'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='043313c7-705a-4015-a6ac-09e5a96bae1d', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='4ca1a1c8-2b54-4c90-b9d9-915fcedc7f7d'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='aabdc665-c535-45c1-abd6-e3ee0762cc63', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='21b6c01c-11ca-42f1-a8f6-a4e1ef03315c'),
              AIMessage(content='Could you please clarify if you mean the device feels unusuall

In [ ]:
print(response["messages"][-1].content)

Could you please clarify if you mean the device feels unusually hot, cold, or if you’re asking how to check its internal temperature? Any additional details will help me assist you better.


Failed to multipart ingest runs: Connection error caused failure to POST https://eu.api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your LANGCHAIN_ENDPOINT. ConnectTimeout(MaxRetryError("HTTPSConnectionPool(host='eu.api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by ConnectTimeoutError(<HTTPSConnection(host='eu.api.smith.langchain.com', port=443) at 0x1f97f3045f0>, 'Connection to eu.api.smith.langchain.com timed out. (connect timeout=3)'))"))
Content-Length: 13755
API Key: lsv2_********************************************39trace=019d7b28-53c0-7821-9acc-aa0993a330ef,id=019d7b28-53c0-7821-9acc-aa0993a330ef; trace=019d7b28-53c0-7821-9acc-aa0993a330ef,id=019d7b28-53c2-7413-9385-07f2e6a86ea3; trace=019d7b28-53c0-7821-9acc-aa0993a330ef,id=019d7b28-53c3-7402-bb55-fef1e75110a3; trace=019d7b28-53c0-7821-9acc-aa0993a330ef,id=019d7b28-53c4-7490-a521-5fbe37ce46a3
Failed to multipart ingest runs: Connection error caused failure to 